# Structured Extraction from Customer Complaints with an LLM

Prompting Gemini to turn free-text complaints into JSON (name, product, category, urgency) and loading the result into a DataFrame.

_Datasets are not included in this repository._

In [9]:
from google.colab import userdata

In [11]:
from google import genai

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
model_info = client.models.get(model="gemini-3.5-flash-lite")
print(model_info)

name='models/gemini-3.5-flash-lite' display_name='Gemini 3.5 Flash Lite' description='Gemini 3.5 Flash Lite' version='3.5-flash-lite-07-2026' endpoints=None labels=None tuned_model_info=TunedModelInfo() input_token_limit=1048576 output_token_limit=65536 supported_actions=['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent'] default_checkpoint_id=None checkpoints=None temperature=1.0 max_temperature=2.0 top_p=0.95 top_k=64 thinking=True


In [12]:
import pandas as pd

In [13]:
data_keluhan = [
    "Halo admin, nama saya Budi Santoso. Laptop saya layarnya tiba-tiba mati total. Ini sangat mendesak karena besok ada presentasi!",
    "Siang, saya Siti Aminah. Resi pengiriman mouse saya tidak bisa dilacak, barang belum sampai juga. Tolong segera dicek.",
    "Pagi, dengan Andi Wijaya. Aplikasi kasir saya error terus saat mau cetak struk. Mohon bantuannya.",
    "Halo, nama saya Rina Marlina. Baterai HP yang saya beli cepat banget habis. Kecewa banget.",
    "Permisi, saya Joko Anwar. Keyboard mekanikal saya ada tombol yang tidak berfungsi.",
    "Sore, saya Maya Sari. Barang pesanan saya statusnya sudah diterima tapi saya belum terima sama sekali dari kurir!",
    "Halo, nama Dedi Irawan. Saya lupa password akun saya dan link reset tidak masuk ke email. Tolong segera dibantu.",
    "Pagi admin, saya Linda Kusuma. Headset bluetooth saya tidak mau pairing ke laptop.",
    "Siang, saya Reza Rahadian. Tagihan saya double di aplikasi, tolong dicek sistemnya.",
    "Halo, nama saya Dewi Lestari. Paket monitor saya datang tapi kardusnya hancur lebur.",
    "Sore, saya Ahmad Fauzi. Kamera webcam saya gambarnya buram dan patah-patah.",
    "Permisi, saya Nina Safitri. Kenapa aplikasi selalu force close saat saya mau upload foto?",
    "Pagi, dengan Rudi Hermawan. Kipas laptop saya bunyinya berisik sekali seperti mesin traktor.",
    "Halo, nama saya Sari Widiastuti. Saya mau retur barang karena salah kirim warna, kurirnya salah alamat.",
    "Siang, saya Eka Putra. Fitur laporan bulanan di aplikasi tidak bisa di-download.",
    "Sore admin, saya Dian Sastro. Layar tablet saya bergaris hijau di tengah.",
    "Halo, saya Gilang Dirga. Pengiriman router saya terlambat 5 hari dari estimasi.",
    "Pagi, nama saya Fitri Tropica. Sistem pembayaran error terus saat pakai kartu kredit.",
    "Siang, saya Tora Sudiro. Touchpad laptop saya kadang loncat-loncat sendiri.",
    "Halo, nama saya Ayu TingTing. Status pengiriman gagal karena rumah kosong, padahal saya ada di rumah seharian!"
]

In [14]:
df = pd.DataFrame(data_keluhan, columns=['keluhan'])

In [15]:
df

,keluhan
0,"Halo admin, nama saya Budi Santoso. Laptop say..."
1,"Siang, saya Siti Aminah. Resi pengiriman mouse..."
2,"Pagi, dengan Andi Wijaya. Aplikasi kasir saya ..."
3,"Halo, nama saya Rina Marlina. Baterai HP yang ..."
4,"Permisi, saya Joko Anwar. Keyboard mekanikal s..."
5,"Sore, saya Maya Sari. Barang pesanan saya stat..."
6,"Halo, nama Dedi Irawan. Saya lupa password aku..."
7,"Pagi admin, saya Linda Kusuma. Headset bluetoo..."
8,"Siang, saya Reza Rahadian. Tagihan saya double..."
9,"Halo, nama saya Dewi Lestari. Paket monitor sa..."


In [16]:
df.to_csv('keluhan.csv', index=True)

In [47]:
input="""Ekstrak informasi dari teks keluhan pelanggan berikut ke dalam format JSON.
Setiap objek JSON harus memiliki kunci: nama, produk, kategori, dan urgensi.

Aturan:
- kategori: hanya boleh 'hardware', 'software', atau 'pengiriman'.
- urgensi: hanya boleh 'tinggi', 'sedang', atau 'rendah'."""

In [48]:
myfile = client.files.upload(file="/content/keluhan.csv")

result = client.models.generate_content(
  model="gemini-3.5-flash-lite",
  contents=[
    myfile,
    input,
  ],
  config={
      'response_mime_type': 'application/json',
  }
)

In [50]:
print(result)

sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""[
  {
    "nama": "Budi Santoso",
    "produk": "Laptop",
    "kategori": "hardware",
    "urgensi": "tinggi"
  },
  {
    "nama": "Siti Aminah",
    "produk": "Mouse",
    "kategori": "pengiriman",
    "urgensi": "tinggi"
  },
  {
    "nama": "Andi Wijaya",
    "produk": "Aplikasi kasir",
    "kategori": "software",
    "urgensi": "tinggi"
  },
  {
    "nama": "Rina Marlina",
    "produk": "Baterai HP",
    "kategori": "hardware",
    "urgensi": "sedang"
  },
  {
    "nama": "Joko Anwar",
    "produk": "Keyboard mekanikal",
    "kategori": "hardware",
    "urgensi": "sedang"
  },
  {
    "nama": "Maya Sari",
    "produk": "Barang pesanan",
    "kategori": "pengiriman",
    "urgensi": "tinggi"
  },
  {
    "nama": "Dedi Irawan",
    "produk": "Akun",
    "kategori": "software",
    "urgensi": "tinggi"
  },
  {
    "nama": "Linda Kusuma",
    "produ

In [51]:
import json

In [52]:
new = json.loads(result.text)

In [53]:
new

[{'nama': 'Budi Santoso',
  'produk': 'Laptop',
  'kategori': 'hardware',
  'urgensi': 'tinggi'},
 {'nama': 'Siti Aminah',
  'produk': 'Mouse',
  'kategori': 'pengiriman',
  'urgensi': 'tinggi'},
 {'nama': 'Andi Wijaya',
  'produk': 'Aplikasi kasir',
  'kategori': 'software',
  'urgensi': 'tinggi'},
 {'nama': 'Rina Marlina',
  'produk': 'Baterai HP',
  'kategori': 'hardware',
  'urgensi': 'sedang'},
 {'nama': 'Joko Anwar',
  'produk': 'Keyboard mekanikal',
  'kategori': 'hardware',
  'urgensi': 'sedang'},
 {'nama': 'Maya Sari',
  'produk': 'Barang pesanan',
  'kategori': 'pengiriman',
  'urgensi': 'tinggi'},
 {'nama': 'Dedi Irawan',
  'produk': 'Akun',
  'kategori': 'software',
  'urgensi': 'tinggi'},
 {'nama': 'Linda Kusuma',
  'produk': 'Headset bluetooth',
  'kategori': 'hardware',
  'urgensi': 'sedang'},
 {'nama': 'Reza Rahadian',
  'produk': 'Tagihan aplikasi',
  'kategori': 'software',
  'urgensi': 'tinggi'},
 {'nama': 'Dewi Lestari',
  'produk': 'Monitor',
  'kategori': 'pengiri

In [54]:
new = pd.DataFrame(new,columns=['nama','produk','kategori','urgensi'])

In [55]:
new

,nama,produk,kategori,urgensi
0,Budi Santoso,Laptop,hardware,tinggi
1,Siti Aminah,Mouse,pengiriman,tinggi
2,Andi Wijaya,Aplikasi kasir,software,tinggi
3,Rina Marlina,Baterai HP,hardware,sedang
4,Joko Anwar,Keyboard mekanikal,hardware,sedang
5,Maya Sari,Barang pesanan,pengiriman,tinggi
6,Dedi Irawan,Akun,software,tinggi
7,Linda Kusuma,Headset bluetooth,hardware,sedang
8,Reza Rahadian,Tagihan aplikasi,software,tinggi
9,Dewi Lestari,Monitor,pengiriman,tinggi
